# Detector benchmark under the fixed-epoch protocol: official split vs HBFMID-Clean

Gate C failed (see docs/RESULTS_GATE_C.md), so the paper rests on the leakage-free benchmark. This notebook adds detectors, each trained 100 epochs (no early stopping, last.pt) on the official split AND on the 5 clean folds:
* **PART = "A"**: YOLO11s, seeds 42/43/44 (3 official + 15 clean runs, about 4 h on T4 x2)
* **PART = "B"**: YOLOv8m seed 42 (1 official + 5 clean runs) and YOLOv8s official split seeds 42/43/44 (the clean YOLOv8s runs already exist); about 4 h
Run A and B as two separate Save Versions (e.g. on two accounts). HBFMID dataset only. Settings: GPU T4 x2, Internet ON (downloads the pretrained weights).

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/nithoukhan1/hbfmid.git"
REPO = "/kaggle/working/hbfmid"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.215"], check=True)
os.environ["WANDB_MODE"] = "disabled"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2, then restart."
print("repo commit:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
DATA_ROOT = "/kaggle/input/datasets/nithoukhan2/hbfmid/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection"
assert os.path.isdir(DATA_ROOT), DATA_ROOT
SMOKE = False
PART = "A"                     # "A" or "B"
JOBS = {"A": [("yolo11s.pt", [42, 43, 44], [42, 43, 44])],            # (model, official seeds, clean seeds)
        "B": [("yolov8m.pt", [42], [42]), ("yolov8s.pt", [42, 43, 44], [])]}[PART]
FOLDS = [0, 1, 2, 3, 4]
if SMOKE:
    EPOCHS, OUT = 2, f"/kaggle/working/results/bench_{PART}_smoke"
    JOBS = [(m, o[:1], c[:1]) for m, o, c in JOBS][:1]; FOLDS = [0]
else:
    EPOCHS, OUT = 100, f"/kaggle/working/results/bench_{PART}"

def make_cmd(model, mode, fold, seed):
    def f(gpu):
        return [sys.executable, f"{REPO}/scripts/training/run_baseline.py", "--mode", mode, "--fold", str(fold), "--seed", str(seed),
                "--model", model, "--epochs", str(EPOCHS), "--fixed-epochs", "--device", str(gpu), "--data-root", DATA_ROOT,
                "--splits-dir", f"{REPO}/data_splits", "--out", OUT]
    return f
TASKS = []
for model, off_seeds, clean_seeds in JOBS:                      # clean runs first (longest), official runs fill the gaps
    TASKS += [(f"clean_f{f}_s{s}_{model[:-3]}", make_cmd(model, "clean", f, s), None) for s in clean_seeds for f in FOLDS]
    TASKS += [(f"official_s{s}_{model[:-3]}", make_cmd(model, "official", 0, s), None) for s in off_seeds]
print(len(TASKS), "runs, epochs =", EPOCHS, "->", OUT)

In [ ]:
import queue, threading, time
os.makedirs(f"{OUT}/logs", exist_ok=True)
gpus = list(range(torch.cuda.device_count()))
q = queue.Queue()
for t in TASKS:
    q.put(t)
failed = []

def worker(gpu):
    while True:
        try:
            name, cmd_fn, needs = q.get_nowait()
        except queue.Empty:
            return
        while needs and not os.path.exists(needs):          # e.g. arm 6 waits for the FracAtlas pre-trained checkpoint
            if any(n.startswith("fa_pre_") for n in failed):
                break
            time.sleep(20)
        t0 = time.time()
        with open(f"{OUT}/logs/{name}.log", "w") as f:
            rc = subprocess.run(cmd_fn(gpu), stdout=f, stderr=subprocess.STDOUT).returncode
        print(f"[GPU {gpu}] {name}: {'OK' if rc == 0 else 'FAILED rc=' + str(rc)} in {(time.time()-t0)/60:.1f} min", flush=True)
        if rc != 0:
            failed.append(name)

threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
[t.start() for t in threads]; [t.join() for t in threads]
print("failed runs:", failed or "none")
for n in failed:
    print("\n==== tail of", n)
    print("".join(open(f"{OUT}/logs/{n}.log").readlines()[-25:]))

In [ ]:
import shutil
subprocess.run([sys.executable, f"{REPO}/scripts/evaluation/aggregate_benchmark.py", "--runs", OUT, f"{REPO}/results/controls", f"{REPO}/results/baseline_fixed",
                "--out", f"{OUT}/benchmark_part{PART}"])
shutil.make_archive(f"/kaggle/working/bench_{PART}_results", "zip", OUT)
print(sorted(os.listdir(OUT)))
if failed:
    print("\nWARNING: failed runs:", failed)

After the run: download `bench_<PART>_results.zip`, unzip once into `results/bench_<PART>/`, commit and push.